In [246]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")




In [247]:
df = pd.read_csv("fertilizer_dataset.csv")
df = df.rename(columns={
    "Temparature": "Temperature",
    "Phosphorous": "Phosphorus"
})
df.columns = df.columns.str.strip()



In [222]:
df.head()

,Temperature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Phosphorus,Potassium,Fertilizer
0,40,74,24,Loamy,Sugarcane,47,59,54,20-20
1,22,64,57,Clayey,Pulses,55,51,41,20-20
2,21,71,50,Red,Wheat,20,31,32,Urea
3,21,58,63,Red,Wheat,40,57,37,14-35-14
4,39,51,29,Sandy,Pulses,49,32,10,MOP


In [248]:
df.columns

Index(['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type',
       'Nitrogen', 'Phosphorus', 'Potassium', 'Fertilizer'],
      dtype='object')

In [224]:
df

,Temperature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Phosphorus,Potassium,Fertilizer
0,40,74,24,Loamy,Sugarcane,47,59,54,20-20
1,22,64,57,Clayey,Pulses,55,51,41,20-20
2,21,71,50,Red,Wheat,20,31,32,Urea
3,21,58,63,Red,Wheat,40,57,37,14-35-14
4,39,51,29,Sandy,Pulses,49,32,10,MOP
...,...,...,...,...,...,...,...,...,...
2095,33,69,56,Clayey,Cotton,30,41,70,10-26-26
2096,21,30,48,Loamy,Wheat,30,59,50,20-20
2097,31,80,63,Clayey,Sugarcane,55,41,58,20-20
2098,30,82,44,Black,Sugarcane,27,76,22,14-35-14


In [249]:
df["Crop_Type"].value_counts()


Crop_Type
Wheat        369
Maize        350
Cotton       349
Sugarcane    345
Paddy        344
Pulses       343
Name: count, dtype: int64

In [250]:
df.describe()

,Temperature,Humidity,Moisture,Nitrogen,Phosphorus,Potassium
count,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000,2100.000000
mean,28.898095,57.823810,51.662857,35.291429,38.784762,37.547619
std,6.604032,16.169405,18.791848,12.109191,16.003941,16.620247
min,18.000000,30.000000,20.000000,10.000000,10.000000,10.000000
25%,23.000000,44.000000,35.000000,26.000000,30.000000,26.000000
50%,29.000000,59.000000,51.000000,35.000000,37.000000,35.000000
75%,34.000000,72.000000,67.000000,43.000000,47.000000,45.000000
max,40.000000,85.000000,85.000000,60.000000,80.000000,80.000000


In [251]:
df.count()

Temperature    2100
Humidity       2100
Moisture       2100
Soil_Type      2100
Crop_Type      2100
Nitrogen       2100
Phosphorus     2100
Potassium      2100
Fertilizer     2100
dtype: int64

In [253]:
X = df.drop("Fertilizer", axis=1)
y = df["Fertilizer"]


In [254]:
X

,Temperature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Phosphorus,Potassium
0,40,74,24,Loamy,Sugarcane,47,59,54
1,22,64,57,Clayey,Pulses,55,51,41
2,21,71,50,Red,Wheat,20,31,32
3,21,58,63,Red,Wheat,40,57,37
4,39,51,29,Sandy,Pulses,49,32,10
...,...,...,...,...,...,...,...,...
2095,33,69,56,Clayey,Cotton,30,41,70
2096,21,30,48,Loamy,Wheat,30,59,50
2097,31,80,63,Clayey,Sugarcane,55,41,58
2098,30,82,44,Black,Sugarcane,27,76,22


In [255]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [256]:


numeric_cols = [
    "Temperature", "Humidity", "Moisture",
    "Nitrogen", "Phosphorus", "Potassium"
]

categorical_cols = ["Crop_Type", "Soil_Type"]

# ----- numeric scaling -----
scaler = StandardScaler()
X_num = pd.DataFrame(
    scaler.fit_transform(df[numeric_cols]),
    columns=numeric_cols
)

# ----- one-hot encode categorical -----
X_cat = pd.get_dummies(
    df[categorical_cols],
    drop_first=False
).astype(int)

# ----- combine -----
X = pd.concat([X_num, X_cat], axis=1)

y = df["Fertilizer"]


In [257]:
from sklearn.preprocessing import LabelEncoder

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)


In [258]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [259]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="multi:softprob",   # multi-class
    num_class=len(le_target.classes_),
    n_estimators=400,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="mlogloss",
    random_state=42
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.08, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None, num_class=7, ...)

In [231]:
X

,Temperature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Phosphorus,Potassium
0,40,74,24,2,4,47,59,54
1,22,64,57,1,3,55,51,41
2,21,71,50,3,5,20,31,32
3,21,58,63,3,5,40,57,37
4,39,51,29,4,3,49,32,10
...,...,...,...,...,...,...,...,...
2095,33,69,56,1,0,30,41,70
2096,21,30,48,2,5,30,59,50
2097,31,80,63,1,4,55,41,58
2098,30,82,44,0,4,27,76,22


In [232]:
le_target = LabelEncoder()
y = le_target.fit_transform(y)


In [260]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le_target.classes_
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

    10-26-26       0.97      0.98      0.98        60
    14-35-14       0.98      0.98      0.98        60
       20-20       0.93      0.87      0.90        60
       28-28       0.92      0.97      0.94        60
         DAP       1.00      1.00      1.00        60
         MOP       1.00      1.00      1.00        60
        Urea       1.00      1.00      1.00        60

    accuracy                           0.97       420
   macro avg       0.97      0.97      0.97       420
weighted avg       0.97      0.97      0.97       420


Confusion Matrix:
[[59  0  1  0  0  0  0]
 [ 0 59  1  0  0  0  0]
 [ 2  1 52  5  0  0  0]
 [ 0  0  2 58  0  0  0]
 [ 0  0  0  0 60  0  0]
 [ 0  0  0  0  0 60  0]
 [ 0  0  0  0  0  0 60]]


In [261]:
import pickle

model_bundle = {
    "model": model,
    "scaler": scaler,
    "feature_columns": X.columns.tolist(),
    "le_target": le_target
}

pickle.dump(model_bundle, open("fertilizer_xgb_model.pkl", "wb"))
print("✅ XGBoost model saved")


✅ XGBoost model saved


Model  1: DEcision tree


In [234]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Decision Tree Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))


Decision Tree Accuracy: 0.9380952380952381
Decision Tree Confusion Matrix:
 [[59  0  1  0  0  0  0]
 [ 0 58  2  0  0  0  0]
 [ 4  2 50  4  0  0  0]
 [ 0  0 11 49  0  0  0]
 [ 0  0  0  0 60  0  0]
 [ 0  0  0  0  0 60  0]
 [ 0  0  0  2  0  0 58]]


MOdel 2: RAndom forest

In [235]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)


rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


Random Forest Accuracy: 0.9619047619047619
Random Forest Confusion Matrix:
 [[60  0  0  0  0  0  0]
 [ 0 60  0  0  0  0  0]
 [ 6  2 46  6  0  0  0]
 [ 0  0  0 59  0  0  1]
 [ 0  0  0  0 60  0  0]
 [ 0  0  0  0  0 60  0]
 [ 0  0  0  1  0  0 59]]


In [236]:
import pickle

with open("fertilizer_model.pkl", "wb") as f:
    pickle.dump({
        "model": rf,
        "le_target": le_target,
        "le_soil": le_soil,
        "le_crop": le_crop
    }, f)


In [237]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_rf, target_names=le_target.classes_))


              precision    recall  f1-score   support

    10-26-26       0.91      1.00      0.95        60
    14-35-14       0.97      1.00      0.98        60
       20-20       1.00      0.77      0.87        60
       28-28       0.89      0.98      0.94        60
         DAP       1.00      1.00      1.00        60
         MOP       1.00      1.00      1.00        60
        Urea       0.98      0.98      0.98        60

    accuracy                           0.96       420
   macro avg       0.96      0.96      0.96       420
weighted avg       0.96      0.96      0.96       420



In [238]:
y_pred = rf.predict(X_test)

# Convert both actual & predicted back to fertilizer names
y_test_names = le_target.inverse_transform(y_test)
y_pred_names = le_target.inverse_transform(y_pred)

# Compare side by side
comparison = pd.DataFrame({
    "Actual Fertilizer": y_test_names,
    "Predicted Fertilizer": y_pred_names
})

comparison


,Actual Fertilizer,Predicted Fertilizer
0,MOP,MOP
1,28-28,28-28
2,28-28,28-28
3,20-20,20-20
4,DAP,DAP
...,...,...
415,28-28,28-28
416,Urea,Urea
417,28-28,28-28
418,14-35-14,14-35-14


In [240]:
X_test = [[
  28, 65, 40,
  le_soil.transform(["Clayey"])[0],
  le_crop.transform(["Wheat"])[0],
  20, 30, 25
]]
dt.predict(X_test)


array([6])